# Chess Coach — an agent for learning chess openings

**Author:** Benoît J. T. GIRARD
**Environment:** Python 3.12, `chess_coach` package (uv)

This notebook walks through the reasoning behind the agent, one step at a time. It
uses the building blocks of the `chess_coach` package directly, so the approach can
be read rather than taken on trust:

0. Configuration and imports
1. Representing a chess position (FEN)
2. Opening theory (local book)
3. Preparing the Wikichess knowledge base (chunking)
4. Generating embeddings
5. The LangGraph agent: orchestrating the tools
6. What the language model is given
7. Summary

> The pure building blocks run as they are. The tools that need external services
> (Stockfish, Milvus, MongoDB) run in the Docker stack; what they do is explained
> here rather than executed.

## 0. Configuration and imports

In [1]:
from chess_coach.config import get_settings
from chess_coach.utils.paths import OPENINGS_DIR, ROOT_DIR, WIKICHESS_DIR

# Where the files are is answered by one module, here as everywhere else: `paths.py` finds
# the project root from a marker file, so this notebook works from a checkout, from a wheel
# and from the container. Nothing below builds a path out of a string.
settings = get_settings()

print("Project root      :", ROOT_DIR.name)
print("Downloaded corpus :", WIKICHESS_DIR.relative_to(ROOT_DIR))
print("Written notes     :", OPENINGS_DIR.relative_to(ROOT_DIR))
print("Embedding model   :", settings.embedding_model)
print("Dimension         :", settings.embedding_dim)
print("Milvus collection :", settings.milvus_collection)

Project root      : backend
Downloaded corpus : var\wikichess
Written notes     : data\openings
Embedding model   : sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Dimension         : 384
Milvus collection : chess_openings


## 1. Representing a chess position (FEN)

Before it can reason, the agent has to read the position. Following the way a
position is handed to a language model (see Kaggle Game Arena), a structured
description is extracted from the FEN: side to move, legal moves, board.

In [2]:
from chess_coach.services.chess_position import STARTING_FEN, describe_position

info = describe_position(STARTING_FEN)
print("Side to move :", info.side_to_move)
print(
    "Legal moves  :",
    len(info.legal_moves_san),
    "->",
    info.legal_moves_san[:6],
    "...",
)
print(info.board_ascii)

Side to move : white
Legal moves  : 20 -> ['Nh3', 'Nf3', 'Nc3', 'Na3', 'h3', 'g3'] ...
r n b q k b n r
p p p p p p p p
. . . . . . . .
. . . . . . . .
. . . . . . . .
. . . . . . . .
P P P P P P P P
R N B Q K B N R


**Observation.** From the starting position the agent has twenty legal moves and a
board it can work with. This description goes to the tools and, where relevant, to
the language model that writes the final answer.

## 2. Opening theory (local book)

For a theoretical move, the agent queries an opening book built from the main lines.
In production the Lichess Opening Explorer enriches those moves with master-game
statistics; the local book is what answers when Lichess is unreachable.

In [3]:
from chess_coach.services.opening_book import OpeningBook

result = OpeningBook.lookup(STARTING_FEN)
print("Opening      :", result.opening_name, f"({result.opening_eco})")
print("Theory moves :", [m.san for m in result.moves])

Opening      : King's Pawn Opening (B00)
Theory moves : ['e4', 'd4', 'c4', 'Nf3']


**Observation.** The book recognises the position and offers the first master moves
(e4, d4, c4, Nf3). Transpositions are handled because positions are indexed by their
EPD, not by the move order that reached them.

## 3. Preparing the knowledge base (chunking)

The corpus sits in two directories: the articles downloaded from Wikichess, the
collaborative chess encyclopaedia hosted by FICGS, and the notes written here to complete
them. The downloaded articles are not committed with this repository, for the licence reason
`docs/data-source.md` sets out, so `python -m scripts.fetch_wikichess` is what puts them on
disk and the cell below reads whatever it finds. How well the retrieval works depends on how
the text is cut, so the articles are loaded first, then segmented into overlapping passages.

In [4]:
from chess_coach.rag.preprocess import build_chunks, load_articles

articles = load_articles(WIKICHESS_DIR) + load_articles(OPENINGS_DIR)
chunks = build_chunks(articles)
print(f"{len(articles)} articles -> {len(chunks)} chunks")
print("Example chunk:")
print(chunks[0].text[:200], "...")

32 articles -> 146 chunks
Example chunk:
King's pawn Opening

Code ECO : B00
Coups : 1.e4
FEN : rnbqkbnr/pppppppp/8/8/4P3/8/PPPP1PPP/RNBQKBNR b KQkq - 0 1
Source : https://ficgs.com/wikichess_1.html
Contributeurs Wikichess : Thibault de Vass ...


**Observation.** Every chunk keeps the name of the opening and the directory it came
from. That is what lets the interface say where a passage comes from: a Wikichess
article, or one of the local notes.

## 4. Generating embeddings

A text becomes a dense, normalised vector. The model is multilingual, and that
matters here: the Wikichess articles are in English, while the local notes and the
questions the user asks are in French.

In [5]:
from chess_coach.services.embeddings import EmbeddingService

# The queries are in French, the articles in English: this is the cross-language case
# the multilingual model is there for.
embedder = EmbeddingService(settings)
vectors = embedder.embed(["La défense sicilienne", "Le gambit dame"])
print("Vectors   :", len(vectors))
print("Dimension :", len(vectors[0]))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Vectors   : 2
Dimension : 384


**Observation.** Each sentence becomes a 384-dimension vector. Indexed in Milvus
(inner-product search, which on normalised vectors is the cosine), they back the
semantic search exposed by `GET /api/v1/vector-search`.

## 5. The LangGraph agent: orchestrating the tools

The agent is a decision graph. For one position:

1. **identify** — describe the position (is the FEN valid?);
2. **theory** — look up the theoretical moves;
3. **branch** — known position goes to theory, otherwise to Stockfish;
4. **context** — add context from the Wikichess retrieval;
5. **videos** — suggest YouTube videos;
6. **synthesize** — write the recommendation, in French;
7. **persist** — record the interaction in MongoDB.

The last step, the synthesis, is illustrated below from a hand-made state, with no
external service running.

In [6]:
from chess_coach.agent.synthesize import build_template_recommendation

# The passages and the answer are in French: French is the language of the product,
# whatever the language of the sources.
state = {
    "fen": "r1bqkbnr/pppp1ppp/2n5/4p3/2B1P3/5N2/PPPP1PPP/RNBQK2R b KQkq - 3 3",
    "in_theory": True,
    "opening_name": "Italian Game",
    "opening_eco": "C50",
    "position": {
        "side_to_move": "black",
        "legal_moves_san": ["Nf6", "Bc5", "d6"],
        "board_ascii": "(diagram)",
    },
    "theory_moves": [
        {"san": "Nf6", "total": 103572421},
        {"san": "Bc5", "total": 101156348},
    ],
    "reference_games": [
        {"white": "Carlsen", "black": "Caruana", "result": "1-0", "year": 2019},
    ],
    "passages": [
        {"opening": "Giuoco Piano", "text": "Le fou en c4 vise la case f7."},
    ],
    "videos": [{"title": "Tutoriel"}],
}

print(build_template_recommendation(state))

Cette position relève de **Italian Game** (C50).

Les coups théoriques principaux sont : Nf6, Bc5.

Pour approfondir : Le fou en c4 vise la case f7.

1 vidéo(s) explicative(s) vous sont proposées ci-dessous.


**Observation.** From the facts the nodes collected, the template alone already
produces a readable answer. This is the agent's safety net: it works without any API
key, and it is what answers when the model call fails.

## 6. What the language model is given

The last node hands the writing to a model. It is not given the bare FEN: as in the
Kaggle Game Arena games, the position is described to it — board, side to move, legal
moves — and the facts gathered by the earlier nodes are added to that.

In [7]:
from chess_coach.agent.synthesize import build_llm_prompt

print(build_llm_prompt(state))

FEN : r1bqkbnr/pppp1ppp/2n5/4p3/2B1P3/5N2/PPPP1PPP/RNBQK2R b KQkq - 3 3
Trait aux : black
Coups légaux : Nf6, Bc5, d6
Échiquier :
(diagram)

Ouverture : Italian Game (C50)
Coups théoriques, avec le nombre de parties de maîtres :
  - Nf6 (103572421 parties)
  - Bc5 (101156348 parties)
Coup mis en avant dans l'interface : Nf6, le plus joué en parties de maîtres.
Parties de référence :
  - Carlsen - Caruana, 1-0, 2019
Le moteur d'analyse n'a pas été sollicité : la position est dans la théorie.

Extraits de la base de connaissances :
  - (Giuoco Piano) Le fou en c4 vise la case f7.

Des vidéos explicatives sont affichées sous ta réponse : tu peux y renvoyer le joueur, sans citer de titre.


**Observation.** The model is meant to write up, not to decide: the moves come from
Lichess, the evaluation from Stockfish, the context from Milvus, and the prompt asks
it to add nothing of its own. Nothing in the test suite checks that it obeys — such a
check would spend money on every run — so this is a design intent, not a measured
property. If the call fails, the template takes over and the agent still answers.

## 7. Summary

- The position is **read**: the FEN becomes a structured description.
- **Theory** and **reference games** come from the Lichess Opening Explorer; the
  local book takes over when it is unreachable.
- Outside theory, **Stockfish** evaluates the position.
- The **Milvus retrieval** over Wikichess brings the context.
- The **YouTube API** suggests videos.
- A **language model** writes the recommendation from those facts.

The whole is orchestrated by **LangGraph**, exposed by **FastAPI**, persisted in
**MongoDB** and presented by the **Angular** interface.